In [23]:
!pip install scapy

mambajs 0.19.13

Process pip requirements ...

Requirement scapy already satisfied.


In [24]:
!mamba install pandas

mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 1.6045 seconds
All requested packages already installed.


In [25]:
!mamba install scikit-learn

mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 1.195 seconds
All requested packages already installed.


In [18]:
from scapy.all import sniff, IP, TCP, UDP
import pandas as pd
import time
import sklearn
from sklearn.ensemble import IsolationForest

In [26]:


# Dictionary to hold our flow statistics
flows = {}

def get_flow_id(packet):
    """Creates a unique string ID for a network connection."""
    if IP in packet:
        src_ip = packet[IP].src
        dst_ip = packet[IP].dst
        proto = packet[IP].proto
        
        src_port, dst_port = 0, 0
        if TCP in packet:
            src_port = packet[TCP].sport
            dst_port = packet[TCP].dport
        elif UDP in packet:
            src_port = packet[UDP].sport
            dst_port = packet[UDP].dport
            
        # Sort so A->B and B->A are treated as the same flow
        endpoints = sorted([f"{src_ip}:{src_port}", f"{dst_ip}:{dst_port}"])
        return f"{endpoints[0]} <-> {endpoints[1]} [{proto}]"
    return None

def process_offline_packet(packet):
    """Analyzes a single packet from the PCAP file."""
    flow_id = get_flow_id(packet)
    
    if flow_id:
        # If it's a new flow, initialize its stats
        if flow_id not in flows:
            flows[flow_id] = {
                'packet_count': 0,
                'total_bytes': 0,
                'start_time': float(packet.time), # Use packet's timestamp, not current time!
                'end_time': float(packet.time)
            }
            
        # Update the stats
        flows[flow_id]['packet_count'] += 1
        flows[flow_id]['total_bytes'] += len(packet)
        flows[flow_id]['end_time'] = float(packet.time)

# --- Main Execution ---

# 1. Provide the path to your Wireshark file here
PCAP_FILE = "wireshark_cap_02.pcap"

print(f"Reading {PCAP_FILE}...")
try:
    # sniff() can read offline files! store=0 keeps RAM usage low.
    sniff(offline=PCAP_FILE, prn=process_offline_packet, store=0)
    print("PCAP processing complete. Calculating AI features...")
    
    # 2. Translate raw stats into AI features
    ai_data = []
    for flow_id, stats in flows.items():
        duration = stats['end_time'] - stats['start_time']
        if duration == 0:
            duration = 0.001 # Prevent division by zero
            
        ai_data.append({
            'Flow ID': flow_id,
            'Packets': stats['packet_count'],
            'Bytes': stats['total_bytes'],
            'Duration (s)': round(duration, 4),
            'Bytes/Sec': round(stats['total_bytes'] / duration, 2),
            'Pkts/Sec': round(stats['packet_count'] / duration, 2),
            'Avg Packet Size': round(stats['total_bytes'] / stats['packet_count'], 2)
        })
        
    # 3. Create the Pandas DataFrame
    df = pd.DataFrame(ai_data)
    
    # Display the top 5 largest flows
    print("\nTop 5 Flows by Byte Count:")
    display(df.sort_values(by="Bytes", ascending=False).head(5)) # 'display()' looks great in Jupyter

except FileNotFoundError:
    print(f"\n[ERROR] Could not find the file '{PCAP_FILE}'.")
    print("Please capture some traffic in Wireshark, save it as a .pcap, and put it in the same folder as this notebook.")

Reading wireshark_cap_02.pcap...
PCAP processing complete. Calculating AI features...

Top 5 Flows by Byte Count:


,Flow ID,Packets,Bytes,Duration (s),Bytes/Sec,Pkts/Sec,Avg Packet Size
16124,10.74.42.143:45330 <-> 52.239.175.4:443 [6],72,407803,0.3599,1133100.87,200.06,5663.93
16088,10.74.42.143:19297 <-> 52.113.194.132:443 [6],88,89876,0.3089,290921.88,284.85,1021.32
16123,10.74.42.143:45329 <-> 20.60.194.225:443 [6],32,87563,0.3420,256004.50,93.56,2736.34
16111,10.74.42.143:45323 <-> 40.126.7.35:443 [6],51,36608,0.4202,87123.36,121.37,717.80
16078,10.74.42.143:19288 <-> 10.74.42.147:443 [6],47,35465,0.6468,54832.59,72.67,754.57


In [27]:

print("Initializing AI Anomaly Detector...")

# 1. Isolate the exact math features the AI needs to look at.
# We drop 'Flow ID' because the AI only understands numbers, not text.
features_for_ai = df[[
    'Packets', 
    'Bytes', 
    'Duration (s)', 
    'Bytes/Sec', 
    'Pkts/Sec', 
    'Avg Packet Size'
]]

# 2. Build the Model (The Brain)
# contamination=0.05 means we are telling the AI: "Assume about 5% of this traffic is weird/anomalous."
print("Training the AI on your network data...")
ai_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)

# 3. Train the model on your data! 
ai_model.fit(features_for_ai)
print("Training complete!")

# 4. Ask the AI to score the traffic it just learned from.
# Isolation Forest returns: 1 for "Normal" and -1 for "Anomaly"
predictions = ai_model.predict(features_for_ai)

# 5. Attach the results back to our readable DataFrame
df['AI_Verdict'] = predictions

# Convert the 1 and -1 into human-readable labels
df['Status'] = df['AI_Verdict'].map({1: 'Normal', -1: '🚨 ANOMALY'})

# 6. Show the results! Let's look at the anomalies first.
anomalies = df[df['Status'] == '🚨 ANOMALY']

print("\n" + "="*60)
print(f"TAMASKAN AI: Found {len(anomalies)} anomalies out of {len(df)} total flows.")
print("="*60)

if not anomalies.empty:
    # Display the anomalous flows
    display(anomalies[['Flow ID', 'Packets', 'Bytes', 'Avg Packet Size', 'Status']].sort_values(by='Bytes', ascending=False))
else:
    print("No anomalies detected! All traffic looks uniform.")

Initializing AI Anomaly Detector...
Training the AI on your network data...
Training complete!

TAMASKAN AI: Found 810 anomalies out of 16190 total flows.


,Flow ID,Packets,Bytes,Avg Packet Size,Status
16124,10.74.42.143:45330 <-> 52.239.175.4:443 [6],72,407803,5663.93,🚨 ANOMALY
16088,10.74.42.143:19297 <-> 52.113.194.132:443 [6],88,89876,1021.32,🚨 ANOMALY
16123,10.74.42.143:45329 <-> 20.60.194.225:443 [6],32,87563,2736.34,🚨 ANOMALY
16111,10.74.42.143:45323 <-> 40.126.7.35:443 [6],51,36608,717.80,🚨 ANOMALY
16078,10.74.42.143:19288 <-> 10.74.42.147:443 [6],47,35465,754.57,🚨 ANOMALY
...,...,...,...,...,...
1744,10.74.42.143:43552 <-> 10.74.42.147:8099 [6],2,118,59.00,🚨 ANOMALY
1659,10.74.42.143:43552 <-> 10.74.42.147:6006 [6],2,118,59.00,🚨 ANOMALY
1652,10.74.42.143:43552 <-> 10.74.42.20:9091 [6],2,118,59.00,🚨 ANOMALY
1552,10.74.42.143:43552 <-> 10.74.42.20:1132 [6],2,118,59.00,🚨 ANOMALY
